In [ ]:
import os
import shutil
from glob import glob
from pathlib import Path

import pandas as pd
from tqdm import tqdm

In [ ]:
human = glob('human_images/[!_]*')
animal = glob('animal_images/[!_]*')

In [ ]:
h = [Path(x).name for x in human]
a = [Path(x).name for x in animal]
assert not [x for x in a if x not in h]

In [ ]:
Path('pilot_mnt_data/transects').mkdir(exist_ok=True, parents=True)
Path('pilot_mnt_data/results').mkdir(exist_ok=True)

In [ ]:
for deployment in h:
    for subfolder in ['calibration_frames', 'calibration_frames_masks', 'detection_frames']:
        Path(f'pilot_mnt_data/transects/{deployment}/{subfolder}').mkdir(exist_ok=True, parents=True)

In [ ]:
for deployment in human:
    for image in glob(f'{deployment}/*'):
        out_path = f'pilot_mnt_data/transects/{Path(deployment).name}/calibration_frames'
        if Path(f'{out_path}/{image}').exists():
            continue
        shutil.copy2(image, out_path)

In [ ]:
for deployment in animal:
    for image in glob(f'{deployment}/*'):
        out_path = f'pilot_mnt_data/transects/{Path(deployment).name}/detection_frames'
        if Path(f'{out_path}/{image}').exists():
            continue
        shutil.copy2(image, out_path)

In [ ]:
%matplotlib inline
from ultralytics import YOLO
from IPython.display import display, Image
import cv2
import numpy as np
import matplotlib.pyplot as plt
from loguru import logger

device = "cuda"

In [ ]:
def debug_fill_mask(binary_mask):
    # Ensure binary_mask is of type uint8
    binary_mask_uint8 = binary_mask.astype(np.uint8)
    
    # Find contours in the binary mask
    contours, _ = cv2.findContours(binary_mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    print(f"Number of contours found: {len(contours)}")  # Debug print
    
    # Create an all zero mask to draw the contours on, ensure it is of type uint8
    contour_filled_mask = np.zeros_like(binary_mask_uint8)
    
    # Fill the contours - this effectively "closes" the holes in the original mask
    cv2.drawContours(contour_filled_mask, contours, -1, 1, thickness=cv2.FILLED)
    
    # Apply morphological operations to close small gaps in the contours
    kernel = np.ones((5,5),np.uint8)
    contour_filled_mask = cv2.morphologyEx(contour_filled_mask, cv2.MORPH_CLOSE, kernel)
    
    return contour_filled_mask.astype(bool)

def visualize_contours(image, binary_mask):
    contours, _ = cv2.findContours(binary_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contour_image = image.copy()
    cv2.drawContours(contour_image, contours, -1, (255,0,0), 3)
    plt.imshow(contour_image)
    plt.axis('off')
    plt.show()


In [ ]:
transects = glob('pilot_mnt_data/transects/*')
# selected_transects = [x for x in transects if len(glob(f'{x}/calibration_frames/*')) == 8]
# len(selected_transects)
transects

In [ ]:
# for transect in transects:
#     if transect not in selected_transects:
#         shutil.rmtree(transect)

In [ ]:
model = YOLO('yolov8x.pt')

In [ ]:
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor

sam_checkpoint = "sam_vit_h_4b8939.pth"
model_type = "vit_h"
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)
predictor = SamPredictor(sam)

In [ ]:
def calculate_area(bbox):
    xmin, ymin, xmax, ymax = bbox
    width = xmax - xmin
    height = ymax - ymin
    area = width * height
    return area

In [ ]:
def get_mask_image(im_path):
    results = model(im_path, classes=[0])
    boxes = results[0].boxes.xyxy.tolist()
    if not boxes:
        logger.error(f'Could not find a human in {im_path}!')
        return im_path
    if len(boxes) > 1:
        bbox1_area = calculate_area(boxes[0])
        bbox2_area = calculate_area(boxes[1])
        if bbox1_area > bbox2_area:
            idx = 1
        elif bbox2_area > bbox1_area:
            idx = 0
    else:
        idx = 0

    if im_path in switch_index:
        idx = idx + 1 if idx == 0 else idx - 1
        if len(boxes) != 2:
            logger.error(f'Could not find calib-human in {im_path}!')
            return im_path

    bbox = boxes[idx]


    image = cv2.cvtColor(cv2.imread(im_path), cv2.COLOR_BGR2RGB)
    predictor.set_image(image)
    
    input_box = np.array(bbox)
    masks, _, _ = predictor.predict(
        point_coords=None,
        point_labels=None,
        box=input_box,
        multimask_output=False,
    )

    segmentation_mask = masks[0]
    
    # Convert the segmentation mask to a binary mask
    binary_mask = np.where(segmentation_mask > 0.5, 1, 0)
    # binary_mask = debug_fill_mask(binary_mask)
    visualize_contours(image, binary_mask)
    # binary_mask = fill_mask(binary_mask) # <<<<<<<<<<<<
    
    # Create white background and black background
    white_background = np.ones_like(image) * 255
    black_background = np.zeros_like(image)
    
    # Apply the binary mask to the white background
    new_image = white_background * binary_mask[..., np.newaxis] + black_background * (1 - binary_mask[..., np.newaxis])
    
    # Display the image
    plt.imsave(f'{deployment_path}/calibration_frames_masks/{Path(im_path).name}', new_image.astype(np.uint8))
    plt.imshow(new_image.astype(np.uint8))
    plt.axis('off')
    plt.show()

In [ ]:
switch_index = ['pilot_mnt_data/transects/PM3 02-10-2024/calibration_frames/18.JPG',
 'pilot_mnt_data/transects/PM3 02-10-2024/calibration_frames/21.JPG',
 'pilot_mnt_data/transects/PM3 02-10-2024/calibration_frames/24.JPG',
 'pilot_mnt_data/transects/Cam23/calibration_frames/15.JPG',
 'pilot_mnt_data/transects/Cam23/calibration_frames/21.JPG',
 'pilot_mnt_data/transects/cam12 02-10-2024/calibration_frames/09.JPG']

In [ ]:
failed_images = []

for deployment_path in transects:
    ims = glob(f'{deployment_path}/calibration_frames/*')
    ims.sort()
    for im_path in ims:
        print(im_path)
        res = get_mask_image(im_path)
        if res:
            failed_images.append(res)
        print('-' * 100)

In [ ]:
for x in failed_images:
    os.remove(x)

In [ ]:
failed_images

In [ ]:
# transects_with_bbox_issues = [
#     'pilot_mnt_data/transects/cam26 02-10-2024',
#     'pilot_mnt_data/transects/Cam6  02-10-2024'
#     'pilot_mnt_data/transects/PM22 02-10-2024/', 
#     'pilot_mnt_data/transects/cam23 02-10-2024'
# ]
# for transect in transects_with_bbox_issues:
#     try:
#         shutil.rmtree(transect)
#     except:
#         continue

In [ ]:
# print(len(selected_transects))
# selected_transects = [x for x in selected_transects if x not in transects_with_bbox_issues]
# print(len(selected_transects))

In [ ]:
# def get_bbox_image(image_path):
#     results = model.predict(source=image_path, classes=[0])
#     boxes = results[0].boxes.xyxy.tolist()
#     if not boxes:
#         logger.error(f'Could not find a human in {image_path}!')
#         return image_path
#     if len(boxes) > 1:
#         bbox1_area = calculate_area(boxes[0])
#         bbox2_area = calculate_area(boxes[1])
#         if bbox1_area > bbox2_area:
#             idx = 1
#         elif bbox2_area > bbox1_area:
#             idx = 0
#     else:
#         idx = 0

#     if image_path in switch_index:
#         idx = idx + 1 if idx == 0 else idx - 1
#         if len(boxes) != 2:
#             logger.error(f'Could not find calib-human in {image_path}!')
#             return image_path

#     bbox = boxes[idx]

#     image = cv2.imread(image_path)
#     mask = np.zeros_like(image)
#     x1, y1, x2, y2 = [int(coord) for coord in bbox]
#     mask[y1:y2, x1:x2] = [255, 255, 255]
#     output_image_path = f'{deployment_path}/calibration_frames_masks/{Path(im_path).name}'
#     cv2.imwrite(output_image_path, mask)

#     cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 3)
#     # Convert BGR to RGB for matplotlib display since OpenCV uses BGR by default.
#     image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
#     # Display the image inline in Jupyter.
#     plt.figure(figsize=(10, 10))  # You can adjust the figure size as needed.
#     plt.imshow(image_rgb)
#     plt.axis('off')  # Turn off axis numbers and ticks.
#     plt.show()

In [ ]:
# import cv2
# import numpy as np
# from matplotlib import pyplot as plt
# from pathlib import Path

# def draw_label_with_background(img, text, position, font, font_scale, text_color, bg_color, font_thickness):
#     # Get the size of the text box
#     text_size = cv2.getTextSize(text, font, font_scale, font_thickness)[0]
#     text_x, text_y = position
#     # Calculate the box coordinates based on the text size
#     box_coords = ((text_x, text_y + 5), (text_x + text_size[0] + 2, text_y - text_size[1] - 2))
#     # Draw the background rectangle and then the text
#     cv2.rectangle(img, box_coords[0], box_coords[1], bg_color, cv2.FILLED)
#     cv2.putText(img, text, (text_x, text_y), font, font_scale, text_color, font_thickness, lineType=cv2.LINE_AA)

# def get_bbox_image(image_path):
#     results = model.predict(source=image_path, classes=[0])
#     boxes = results[0].boxes.xyxy.tolist()
    
#     if not boxes:
#         print(f'Could not find a human in {image_path}!')
#         return

#     image = cv2.imread(image_path)
#     # For displaying all boxes with indices
#     image_with_boxes = image.copy()
    
#     for i, (x1, y1, x2, y2) in enumerate(boxes):
#         # Draw bounding boxes larger and with thicker borders
#         cv2.rectangle(image_with_boxes, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 4)
        
#         # Label text without the '#' symbol and bigger size
#         label = f'{i}'
#         # Calculate position for the label to keep it visible
#         label_x = max(int(x1), 0)
#         label_y = max(int(y1) - 10, 0)
#         label_y = min(label_y, image.shape[0] - 20)  # Adjust if too close to bottom
        
#         # Draw label with background for better visibility
#         draw_label_with_background(image_with_boxes, label, (label_x, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), (0,0,0), 3)

#     # Convert BGR to RGB for matplotlib display
#     image_with_boxes_rgb = cv2.cvtColor(image_with_boxes, cv2.COLOR_BGR2RGB)
    
#     plt.figure(figsize=(10, 10))
#     plt.imshow(image_with_boxes_rgb)
#     plt.axis('off')
#     plt.show()
    
#     selected_index = 0  # Default to the first box if only one is present.
#     if len(boxes) > 1:
#         try:
#             selected_index = int(input("Enter the index of the bounding box you want to select: "))
#             if selected_index < 0 or selected_index >= len(boxes):
#                 print("Invalid index. Exiting function.")
#                 return
#         except ValueError:
#             print("Invalid input. Exiting function.")
#             return

#     # Process the selected bounding box as needed...


In [ ]:
# failed_images = []
# for deployment_path in selected_transects:
#     ims = glob(f'{deployment_path}/calibration_frames/*')
#     ims.sort()
#     for im_path in ims:
#         print(im_path)
#         res = get_bbox_image(im_path)
#         if res:
#             failed_images.append(res)
#         print('-' * 100)
# logger.info(f'Number of images with no human detection: {len(failed_images)}')

In [ ]:
# failed_images

In [ ]:
# remove_transects = [
#     'pilot_mnt_data/transects/PM3 02-10-2024',
#     'pilot_mnt_data/transects/Cam5 02-10-2024',
#     'pilot_mnt_data/transects/cam2 02-10-2024',
#     'pilot_mnt_data/transects/Cam4 02-10-2024',
#     'pilot_mnt_data/transects/cam12 02-10-2024',
#     'pilot_mnt_data/transects/PM10'
# ]
    
# for transect in remove_transects:
#     shutil.rmtree(transect)


In [ ]:
# ! ls pilot_mnt_data/transects

In [ ]:
# df = pd.read_csv('wildlife-insights_4fc92e40-94fe-48d3-a51f-5ac1a03f5e03_project-2007160_data/deployments.csv')
# df['deployment_id'] = df['deployment_id'].str.replace('/', '-')

In [ ]:
# df.head()

In [ ]:
# current_transects = glob('pilot_mnt_data/transects/*')
# current_transects_name = [Path(x).name for x in current_transects]
# current_transects_name

In [ ]:
# df = df[df['deployment_id'].isin(current_transects_name)]

In [ ]:
# df['camera_name']

In [ ]:
# df[df['camera_name'] == 'Reconyx Hyperfire 2']